In [1]:
import json
import os
import subprocess
import sys
import threading
from pathlib import Path

import py3Dmol
from sample.sample_config import (
    GenerationParams,
    SampleCheckpointParams,
    SampleConfig,
    SampleOutputParams,
)
from train.train_config import (
    CheckpointParams,
    TrainLoaderConfig,
    TrainConfig,
    TrainingParams,
)

PALLATOM_ROOT = Path.cwd().resolve()
if str(PALLATOM_ROOT) not in sys.path:
    sys.path.insert(0, str(PALLATOM_ROOT))

PALLATOM_ROOT

PosixPath('/workspaces/diffusion/pallatom')

In [2]:
import torch

torch.cuda.is_available()

False

# PallAtom: Training & Analysis

1. **Configure** — tune hyperparameters and serialise `TrainConfig` to JSON
2. **Train** — launch `train_loop.py` as a subprocess
3. **Sample** — load the saved checkpoint and run EDM backbone sampling
4. **Visualise** — render sampled structures with py3Dmol

## 1 · Training Configuration

In [3]:
def run_subprocess(cmd, out: dict):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PALLATOM_ROOT)
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(PALLATOM_ROOT),
        env=env,
    )
    out["pid"] = proc.pid
    print(f"Subprocess PID: {proc.pid}")

    stdout, _ = proc.communicate()
    output_lines = stdout.strip().splitlines() if stdout else []

    if proc.returncode != 0:
        print([f"ERROR (exit {proc.returncode}):"] + output_lines)
        out["result"] = None
        return

    out["result"] = output_lines

In [ ]:
tcfg = TrainConfig(
    training=TrainingParams(
        num_epochs=50,
        # pretrained_weights="pallatom_best.pt"
        ),
    checkpoint=CheckpointParams(checkpoint_path="pallatom_toy_best.pt"),
    train_loader=TrainLoaderConfig(max_seq_length=128, token_budget=256)
    )
tcfg

TrainConfig(training=TrainingParams(num_epochs=50, lr=0.0003, weight_decay=0.0001, grad_clip=2.0, pretrained_weights=None, resume_checkpoint=None, accumulated_token_budget=4096), model=ModelParams(window_size=32, f_ref_dim=35, c_atom=4, c_pair=4, c_res=8, c_atompair=2, K_unit=8, max_residues=128, n_amino=20, n_blocks_atom_transformer_encoder=3, n_heads_atom_transformer_encoder=4, n_blocks_atom_transformer_decoder=3, n_heads_atom_transformer_decoder=4, n_pairformer_blocks_template_embedder=2, n_paiformer_heads_template_embedder=16), noise=NoiseScheduleParams(sigma_data=16.0, sigma_max=160, sigma_min=0.0004, P_mean=-1.2, P_std=1.5), distogram_res=ResidueDistogramParams(min_dist=3.25, max_dist=50.75, n_bins=39, tok_emb_dim=32), distogram_atom=AtomDistogramParams(min_dist=0.0, max_dist=10.0, n_bins=22, tok_emb_dim=32), loss=LossParams(lam=1.0, alpha_0=0.25, alpha_1=1.0, alpha_2=0.5, alpha_3=0.5, alpha_4=1.0, gamma=0.99, smooth_lddt_cutoff=15), checkpoint=CheckpointParams(checkpoint_path=Po

In [5]:
config_json_path = PALLATOM_ROOT / "train" / "run_config.json"
_ = config_json_path.write_text(
    tcfg.model_dump_json(indent=2)
)

In [8]:
log_path = PALLATOM_ROOT / "train" / "train_logs.jsonl"
shard_dir: Path = PALLATOM_ROOT / "data" / "shards"

train_cmd = [
            sys.executable, "-u",
            str(PALLATOM_ROOT / "train" / "train_loop.py"),
            "--dataset_jsonl",        str(PALLATOM_ROOT / "data" / "chain_set.jsonl"),
            "--keys_for_splits_json",      str(PALLATOM_ROOT / "data" / "chain_set_splits.json"),
            "--config",      config_json_path,
            "--structlog_jsonl",    log_path,
            "--debug_run",
            "--shard_dir", shard_dir,
        ]

training_out = dict()
threading.Thread(
    target=run_subprocess,
    args=(train_cmd, training_out),
    daemon=True
).start()

Subprocess PID: 64621


In [ ]:
import torch

NUM_GPUS = torch.cuda.device_count()
torchrun = str(Path(sys.executable).parent / "torchrun")
ddp_log_path = PALLATOM_ROOT / "train" / "train_logs_ddp.jsonl"
shard_dir: Path = PALLATOM_ROOT / "data" / "shards"

ddp_train_cmd = [
    torchrun,
    f"--nproc_per_node={NUM_GPUS}",
    str(PALLATOM_ROOT / "train" / "train_loop.py"),
    "--dataset_jsonl",        str(PALLATOM_ROOT / "data" / "chain_set.jsonl"),
    "--keys_for_splits_json",      str(PALLATOM_ROOT / "data" / "chain_set_splits.json"),
    "--config",      config_json_path,
    "--structlog_jsonl",    log_path,
    "--debug_run",
    "--shard_dir", shard_dir,
    "--ddp",
]

ddp_training_out = dict()
threading.Thread(
    target=run_subprocess,
    args=(ddp_train_cmd, ddp_training_out),
    daemon=True
).start()

Subprocess PID: 63026


['ERROR (exit 1):', 'Traceback (most recent call last):', '  File "/opt/venv/bin/torchrun", line 10, in <module>', '    sys.exit(main())', '  File "/opt/venv/lib/python3.10/site-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 362, in wrapper', '    return f(*args, **kwargs)', '  File "/opt/venv/lib/python3.10/site-packages/torch/distributed/run.py", line 990, in main', '    run(args)', '  File "/opt/venv/lib/python3.10/site-packages/torch/distributed/run.py", line 981, in run', '    elastic_launch(', '  File "/opt/venv/lib/python3.10/site-packages/torch/distributed/launcher/api.py", line 170, in __call__', '    return launch_agent(self._config, self._entrypoint, list(args))', '  File "/opt/venv/lib/python3.10/site-packages/torch/distributed/launcher/api.py", line 279, in launch_agent', '    spec = WorkerSpec(', '  File "<string>", line 19, in __init__', '  File "/opt/venv/lib/python3.10/site-packages/torch/distributed/elastic/agent/server/api.py", line 108,

In [28]:
ckpt_path       = str(PALLATOM_ROOT / tcfg.checkpoint.checkpoint_path)
ckpt_path

'/workspaces/diffusion/pallatom/pallatom_best_best.pt'

In [30]:
sample_output_path     = str(Path(config_json_path).with_name("samples.json"))
sample_cfg_path = str(Path(config_json_path).with_name("sample_config.json"))

scfg = SampleConfig(
    model=tcfg.model,
    noise=tcfg.noise,
    generation=GenerationParams(
        n_res=tcfg.test_loader.max_seq_length,
        n_samples=tcfg.test_loader.batch_size
    ),
    checkpoint=SampleCheckpointParams(checkpoint_path=ckpt_path),
    output=SampleOutputParams(output_path=sample_output_path),
)

sample_config_json_path = PALLATOM_ROOT / "train" / "sample_config.json"
with open(sample_config_json_path, "w") as _f:
    _f.write(json.dumps(scfg.model_dump(), indent=2))

In [31]:
sample_log_path = PALLATOM_ROOT / "train" / "sample_logs.jsonl"
sample_cmd = [
            sys.executable, "-u",
            str(PALLATOM_ROOT / "sample" / "sampling.py"),
            "--config", sample_cfg_path,
            "--log_file", sample_log_path,
        ]
# does this not pipe logs out into a structlog
sample_out = dict()
threading.Thread(
    target=run_subprocess,
    args=(sample_cmd, sample_out),
    daemon=True
).start()

Subprocess PID: 33763


In [33]:
sample_out

{'pid': 33763,
 'result': ['\x1b2026-05-07T22:46:59.654076Z\x1b [\x1b\x1binfo     \x1b[0m] \x1bconfig loaded                 \x1b \x1bconfig\x1b=\x1b/workspaces/diffusion/pallatom/train/sample_config.json\x1b \x1bn_res\x1b=\x1b128\x1b \x1bn_samples\x1b=\x1b2\x1b',
  '\x1b2026-05-07T22:47:00.744293Z\x1b [\x1b\x1binfo     \x1b[0m] \x1bmodel loaded                  \x1b \x1bcheckpoint\x1b=\x1b/workspaces/diffusion/pallatom/pallatom_best_best.pt\x1b \x1bdevice\x1b=\x1bcuda\x1b',
  '\x1b2026-05-07T22:47:01.032438Z\x1b [\x1b\x1binfo     \x1b[0m] \x1bsampling                      \x1b \x1bddim_steps\x1b=\x1b40\x1b \x1bn_res\x1b=\x1b128\x1b \x1bn_samples\x1b=\x1b2\x1b',
  '\x1b2026-05-07T22:47:09.791698Z\x1b [\x1b\x1binfo     \x1b[0m] \x1bsampling complete             \x1b \x1bn_res\x1b=\x1b128\x1b \x1bn_samples\x1b=\x1b2\x1b',
  '\x1b2026-05-07T22:47:09.814506Z\x1b [\x1b\x1binfo     \x1b[0m] \x1boutput written                \x1b \x1bn_structures\x1b=\x1b2\x1b \x1bpath\x1b=\x1b/workspaces/dif

In [34]:
# lets load the pdb files from samples.json

# Open the file in read mode ('r')
with open('train/samples.json') as file:
    # Use json.load() to read and parse the file
    data = json.load(file)

# Now 'data' is a standard Python object (dict or list)
print(len(data))
pdb_str = data[1]
view = py3Dmol.view(
    width=600, height=600, linked=True , viewergrid=(1, 1))
view.setViewStyle({'style': 'outline', 'color': 'black', 'width': 0.1})
style = {"cartoon": {'color': 'spectrum'}}

view.addModelsAsFrames(pdb_str, viewer=(0, 0))
view.setStyle({'model': -1}, style, viewer=(0, 0))
view.zoomTo(viewer=(0, 0))

view.render()


# and then show them in py3dmol

2


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Sample from the API

In [ ]:
import time
import requests

REST_APIS_ROOT = PALLATOM_ROOT.parent / "REST_APIs"
API_HOST = "127.0.0.1"
API_PORT = 8000
API_BASE = f"http://{API_HOST}:{API_PORT}"

# Start the server only if not already running
try:
    requests.get(f"{API_BASE}/health", timeout=1)
    print("Server already running")
except requests.exceptions.ConnectionError:
    env = {**os.environ, "PYTHONPATH": str(PALLATOM_ROOT), "CHECKPOINT_PATH": ckpt_path}
    api_proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "api:app", "--host", API_HOST, "--port", str(API_PORT)],
        cwd=str(REST_APIS_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(f"Server started (PID {api_proc.pid}), waiting for model to load...")

    for _ in range(120):
        time.sleep(1)
        try:
            if requests.get(f"{API_BASE}/health", timeout=2).ok:
                print("Server ready")
                break
        except requests.exceptions.ConnectionError:
            pass
    else:
        raise RuntimeError("Server did not become ready within 120 s")

# Unconditionally Sample
resp = requests.post(
    f"{API_BASE}/sample",
    json={
        "n_res": scfg.generation.n_res,
        "n_samples": scfg.generation.n_samples,
        "ddim_steps": scfg.sampler.ddim_steps,
    },
    timeout=300,
)
resp.raise_for_status()
payload = resp.json()
pdb_strings = payload["pdb_strings"]
print(f"Received {len(pdb_strings)} structures ({payload['n_res']} res each) from {payload['device']}")


conditional sampling from amino acid sequence alone (no templates)

In [ ]:
n_res = scfg.generation.n_res
AA20 = "ACDEFGHIKLMNPQRSTVWY"
example_seq = (AA20 * (n_res // len(AA20) + 1))[:n_res]

resp = requests.post(
    f"{API_BASE}/sample",
    json={
        "n_res": n_res,
        "n_samples": scfg.generation.n_samples,
        "ddim_steps": scfg.sampler.ddim_steps,
        "sequence": example_seq,
    },
    timeout=300,
)
resp.raise_for_status()
seq_cond_pdbs = resp.json()["pdb_strings"]
print(f"[seq only] {len(seq_cond_pdbs)} structures")


sequence + partial template

In [ ]:
def slice_pdb(pdb_str: str, n_residues: int) -> str:
    """Return a PDB string truncated to the first n_residues unique residue numbers."""
    seen: list[int] = []
    kept: list[str] = []
    for line in pdb_str.splitlines():
        if line.startswith("ATOM"):
            res_num = int(line[22:26])
            if res_num not in seen:
                seen.append(res_num)
            if len(seen) > n_residues:
                continue
        kept.append(line)
    return "\n".join(kept)

partial_pdb = slice_pdb(pdb_strings[0], n_res // 2)

resp = requests.post(
    f"{API_BASE}/sample",
    json={
        "n_res": n_res,
        "n_samples": scfg.generation.n_samples,
        "ddim_steps": scfg.sampler.ddim_steps,
        "sequence": example_seq,
        "template_pdb": partial_pdb,
    },
    timeout=300,
)
resp.raise_for_status()
seq_partial_templ_pdbs = resp.json()["pdb_strings"]
print(f"[seq + partial template] {len(seq_partial_templ_pdbs)} structures")


no sequence + partial template

In [ ]:
resp = requests.post(
    f"{API_BASE}/sample",
    json={
        "n_res": n_res,
        "n_samples": scfg.generation.n_samples,
        "ddim_steps": scfg.sampler.ddim_steps,
        "template_pdb": partial_pdb,
    },
    timeout=300,
)
resp.raise_for_status()
partial_templ_pdbs = resp.json()["pdb_strings"]
print(f"[partial template only] {len(partial_templ_pdbs)} structures")

no sequence + full template

In [ ]:
resp = requests.post(
    f"{API_BASE}/sample",
    json={
        "n_res": n_res,
        "n_samples": scfg.generation.n_samples,
        "ddim_steps": scfg.sampler.ddim_steps,
        "template_pdb": pdb_strings[0],
    },
    timeout=300,
)
resp.raise_for_status()
full_templ_pdbs = resp.json()["pdb_strings"]
print(f"[full template only] {len(full_templ_pdbs)} structures")
